In [9]:
# notebook for removing markdown from LLM outputs

In [26]:
import glob
import numpy as np
import os
import re
import pandas as pd
import editdistance


In [30]:
def get_text(problem_id, model, prompt_detail, temp, i):
    with open(f"problem{problem_id}/completions_{model}/completion_{prompt_detail}_t{temp}_{i}.v", "r") as f:
        return f.read()

In [29]:
def strip_md(txt):
    # check if there is code markdown
    t = txt
    m = re.search("\`{3,}\n*?.*?\n+([\s\S]*?)\`{3,}", t)
    if m: # if there is a match, this extracts the code block
        t = m.group(1)
    t = t.replace("``` verilog", "")
    t = t.replace("```verilog", "")
    t = t.replace("```", "")
    return t

In [24]:
track = []
for problem_id in range(1, 25):
    for model in ['llm', 'gpt3.5', 'gpt4', 'gemini', 'mistral', 'claude', 'llama']:
        for prompt_detail in ['L', 'M', 'H']:
            for temp in [0.1, 0.3, 0.5, 0.7, 1.0]:
                for i in range(100):
                    # check all files for markdown
                    t_org = get_text(problem_id, model, prompt_detail, temp, i)
                    t_strip = strip_md(t_org)
                    if t_org != t_strip:
                        track.append([problem_id, model, prompt_detail, temp, i, editdistance.eval(t_org, t_strip)])
                        # os.makedirs(f"completions_backup/problem{problem_id}/completions_{model}/", exist_ok=True)
                        # with open(f"completions_backup/problem{problem_id}/completions_{model}/completion_{prompt_detail}_t{temp}_{i}.v", "w") as f:
                        #     f.write(t_org)
                        # with open(f"problem{problem_id}/completions_{model}/completion_{prompt_detail}_t{temp}_{i}.v", "w") as f:
                        #     f.write(t_strip)

In [25]:
track

[]

In [32]:
def strip_md_c(txt):
    # check if there is code markdown
    t = txt
    m = re.search("\`{3,}\n*?.*?\n+([\s\S]*?)\`{3,}", t)
    if m: # if there is a match, this extracts the code block
        t = m.group(1)
    t = t.replace("``` c", "")
    t = t.replace("```c", "")
    t = t.replace("```", "")
    return t

In [46]:
c_track = []
for problem_id in range(25, 26):
    for model in ['gpt3.5', 'gpt4', 'llama', 'claude', 'mistral', 'gemini']:
        for prompt_detail in ['L', 'M', 'H']:
            for temp in [0.1, 0.3, 0.5, 0.7, 1.0]:
                for i in range(100):
                    # check all files for markdown
                    with open(f"problem{problem_id}/completions_{model}/completion_{prompt_detail}_t{temp}_{i}.c", "r") as f:
                        t_org = f.read()
                    t_strip = strip_md_c(t_org)
                    if t_org != t_strip:
                        c_track.append([problem_id, model, prompt_detail, temp, i, editdistance.eval(t_org, t_strip)])
                        # os.makedirs(f"completions_backup/problem{problem_id}/completions_{model}/", exist_ok=True)
                        # with open(f"completions_backup/problem{problem_id}/completions_{model}/completion_{prompt_detail}_t{temp}_{i}.c", "x") as f:
                        #     f.write(t_org)
                        # with open(f"problem{problem_id}/completions_{model}/completion_{prompt_detail}_t{temp}_{i}.c", "w") as f:
                        #     f.write(t_strip)

In [47]:
c_track

[]

In [48]:
org = pd.read_csv("completions_backup/stats.csv")

In [49]:
org

,problem_id,model,prompt_detail,temp,i,levenshtein distance
0,1,gpt3.5,L,0.3,1,14
1,1,gpt3.5,L,0.3,2,14
2,1,gpt3.5,L,0.3,11,14
3,1,gpt3.5,L,0.3,19,14
4,1,gpt3.5,L,0.3,40,14
...,...,...,...,...,...,...
33646,25,mistral,H,1.0,85,7
33647,25,mistral,H,1.0,86,7
33648,25,mistral,H,1.0,94,8
33649,25,mistral,H,1.0,96,8


In [37]:
data = pd.DataFrame(c_track, columns = ['problem_id', 'model', 'prompt_detail', 'temp', 'i', 'levenshtein distance'])
data

,problem_id,model,prompt_detail,temp,i,levenshtein distance
0,25,llama,L,1.0,17,4
1,25,llama,M,0.7,17,4
2,25,llama,M,0.7,29,4
3,25,llama,M,0.7,62,4
4,25,llama,M,1.0,5,4
...,...,...,...,...,...,...
1264,25,mistral,H,1.0,85,7
1265,25,mistral,H,1.0,86,7
1266,25,mistral,H,1.0,94,8
1267,25,mistral,H,1.0,96,8


In [38]:
tot = pd.concat([org, data])

In [39]:
tot

,problem_id,model,prompt_detail,temp,i,levenshtein distance
0,1,gpt3.5,L,0.3,1,14
1,1,gpt3.5,L,0.3,2,14
2,1,gpt3.5,L,0.3,11,14
3,1,gpt3.5,L,0.3,19,14
4,1,gpt3.5,L,0.3,40,14
...,...,...,...,...,...,...
1264,25,mistral,H,1.0,85,7
1265,25,mistral,H,1.0,86,7
1266,25,mistral,H,1.0,94,8
1267,25,mistral,H,1.0,96,8


In [41]:
tot.to_csv("completions_backup/stats.csv", index=False)

In [40]:
tot.drop_duplicates()

,problem_id,model,prompt_detail,temp,i,levenshtein distance
0,1,gpt3.5,L,0.3,1,14
1,1,gpt3.5,L,0.3,2,14
2,1,gpt3.5,L,0.3,11,14
3,1,gpt3.5,L,0.3,19,14
4,1,gpt3.5,L,0.3,40,14
...,...,...,...,...,...,...
1264,25,mistral,H,1.0,85,7
1265,25,mistral,H,1.0,86,7
1266,25,mistral,H,1.0,94,8
1267,25,mistral,H,1.0,96,8


In [40]:
def print_diff(i):
    print(get_text(*track[i][:-1]))
    print("- - " * 30)
    print(strip_md(get_text(*track[i][:-1])))
    print("--- edit distance:", track[i][-1])

In [41]:
print_diff(1010)

```verilog
module quark(
    input clk, 
    input reset, 
    input [0:135] s, 
    output [0:135] daout
);

reg [0:67] X;
reg [0:67] Y;
reg [0:9] L;
reg [0:135] state;

quark_p p(
    .L(L),
    .daout(daout[0:9])
);
quark_f f(
    .X(state[0:67]),
    .daout(daout[10:77])
);
quark_g g(
    .Y(state[68:135]),
    .daout(daout[78:135])
);
quark_h h(
    .X(state[0:67]),
    .Y(state[68:135]),
    .L(L),
    .daout(state[0:135])
);

always @(posedge clk) begin
    if(reset == 0) begin
        X <= s[0:67];
        Y <= s[68:135];
        L <= 10'b1111111111;
    end else begin
        state <= {daout, state[0:125]};
    end
end

endmodule
```
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
module quark(
    input clk, 
    input reset, 
    input [0:135] s, 
    output [0:135] daout
);

reg [0:67] X;
reg [0:67] Y;
reg [0:9] L;
reg [0:135] state;

quark_p p(
    .L(L),
    .daout(daout[0:9])
);
quark_f f(
    .X(st